# Atlas Iconocrático — Tutorial 1: Gestão do Corpus e Sistema de Codificação

**Tese de Doutorado** · PPGD/UFSC  
*Contrato Sexual Visual & Regime Iconocrático: Alegorias Femininas na Cultura Jurídica (1789–2024)*

---

## Sobre este tutorial

Este caderno cobre as duas primeiras etapas operacionais da pesquisa:

1. **Gestão do corpus visual** — como catalogar, organizar e validar as 300–500 imagens de alegorias femininas em suportes oficiais (moedas, selos, monumentos, pinturas, gravuras) produzidas nos cinco países do corpus: França (FR), Reino Unido (UK), Estados Unidos (USA), Alemanha (DE) e Brasil (BR).

2. **Sistema de codificação dos 10 indicadores** — como aplicar o protocolo quantitativo que operacionaliza as categorias analíticas da tese (corporalidade, militância, maternidade, branquitude, etc.).

Ao final, o leitor deverá ser capaz de:
- Criar e popular um catálogo CSV com metadados completos de imagens históricas;
- Entender a lógica, a escala e os critérios de decisão de cada um dos 10 indicadores;
- Produzir codificações válidas e consolidar o dataset para análise estatística.

---

## Pré-requisitos

```bash
# Na raiz do repositório:
pip install -r requirements.txt
```

Este notebook assume que o pacote `atlas_iconocratico` está instalado ou que o diretório raiz do projeto está no `PYTHONPATH`. A célula de inicialização abaixo cuida disso automaticamente.

---

> **Nota para a banca:** todos os dados utilizados neste tutorial são historicamente documentados — as imagens mencionadas existem e as codificações refletem análise iconográfica real. Os caminhos de arquivo gerados durante a execução são temporários (`/tmp/atlas_demo`).


---
## 1. Inicialização do Ambiente

### Estrutura do pacote

O `atlas_iconocratico` é organizado em seis módulos especializados:

| Módulo | Função |
|--------|--------|
| `corpus` | Gestão do catálogo CSV (`corpus_visual_master.csv`), organização por país, validação |
| `codificacao` | Protocolo de codificação dos 10 indicadores, validação de entradas, exportação |
| `estatistica` | Análise descritiva e inferencial das hipóteses H1–H5 |
| `kappa` | Cálculo do coeficiente Kappa de Cohen para confiabilidade inter-codificadores |
| `iconclass_` | Integração com a biblioteca iconográfica Iconclass (BARTSCH/RKD) |
| `matriz` | Consolidação da matriz de evidências e exportação final |

A célula abaixo configura o `sys.path` para que o pacote possa ser importado independentemente de como o Jupyter foi iniciado.


In [ ]:
import sys
import os
from pathlib import Path

# ── Garantir que o diretório raiz do projeto está no PYTHONPATH ──────────────
# Este notebook está em: <raiz>/notebooks/01_corpus_e_codificacao.ipynb
# O pacote está em:      <raiz>/atlas_iconocratico/
notebook_dir = Path(".").resolve()
project_root = notebook_dir.parent  # sobe um nível: notebooks/ → raiz
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Raiz do projeto: {project_root}")
print(f"Python path configurado: OK")

# ── Importações dos módulos do Atlas Iconocrático ────────────────────────────
from atlas_iconocratico.corpus import (
    GerenciadorCorpus,
    ImagemCorpus,
    criar_corpus,
    PAISES,
    SUPORTES,
    REGIMES,
    FIGURAS,
)
from atlas_iconocratico.codificacao import (
    GerenciadorCodificacao,
    CodificacaoImagem,
    INDICADORES,
    descrever_indicador,
    descrever_todos_indicadores,
    gerar_manual_codificacao,
    arvore_decisao,
)

print("\nMódulos importados com sucesso:")
print(f"  → corpus       : GerenciadorCorpus, ImagemCorpus")
print(f"  → codificacao  : GerenciadorCodificacao, CodificacaoImagem, {len(INDICADORES)} indicadores")


In [ ]:
# ── Diretório de trabalho para este tutorial ─────────────────────────────────
# Usamos /tmp/atlas_demo para não poluir o repositório durante a demonstração.
# Em produção, use: DATA_DIR = project_root / "data"

DATA_DIR = Path("/tmp/atlas_demo")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Diretório de dados: {DATA_DIR}")
print(f"  corpus_visual_master.csv → {DATA_DIR / 'corpus_visual_master.csv'}")
print(f"  codificacao_COD1.csv     → {DATA_DIR / 'codificacao_COD1.csv'}")
print(f"  corpus_codificado.csv    → {DATA_DIR / 'corpus_codificado.csv'}")


---
## 2. Gestão do Corpus Visual

### O catálogo `corpus_visual_master.csv`

O `GerenciadorCorpus` mantém um único arquivo CSV como fonte de verdade do corpus.  
Cada linha corresponde a uma imagem e contém os seguintes campos:

| Campo | Tipo | Descrição |
|-------|------|-----------|
| `Image_ID` | `str` (4 dígitos) | Identificador único auto-incrementado (`0001`, `0002`, …) |
| `Country` | `str` | Código ISO do país: `FR`, `UK`, `USA`, `DE`, `BR` |
| `Figure` | `str` | Alegoria representada: `Marianne`, `Britannia`, `Liberty`, `Columbia`, `Germania`, `República`, `Justiça`, `Outra` |
| `Support` | `str` | Suporte material: `moeda`, `selo`, `monumento`, `arquitetura_forense`, `emblema`, `pintura`, `gravura`, `fotografia`, `outro` |
| `Year` | `str` | Ano de criação ou emissão |
| `Source_URL` | `str` | URL permanente da fonte primária (Wikimedia, BNF, etc.) |
| `Rights` | `str` | Status de direitos: `domínio_público`, `fair_use_acadêmico`, `licença_CC`, `outro` |
| `File_path` | `str` | Caminho local no repositório (relativo ao `data/`) |
| `Regime_preliminary` | `str` | Regime iconocrático preliminar: `fundacional`, `normativo`, `militar` |
| `Description` | `str` | Descrição iconográfica breve |
| `Artist` | `str` | Artista/autor, se conhecido |
| `Collection` | `str` | Acervo ou museu de origem |
| `Added_date` | `str` | Data de inclusão no corpus (ISO 8601) |

### Convenção de nomeação de arquivos

O método `imagem.nome_arquivo()` gera nomes padronizados no formato:

```
{PAÍS}_{Figura}_{Ano}_{Suporte}_{ID}.{ext}
```

**Exemplos:**
- `FR_Marianne_1830_pintura_0001.jpg` → *La Liberté guidant le peuple* (Delacroix)
- `UK_Britannia_1860_moeda_0002.jpg` → *Penny britânica vitoriana*
- `BR_República_1889_pintura_0005.jpg` → *Efígie da República* (Henrique Bernardelli)

Esta convenção garante legibilidade humana e ordenação lexicográfica correta no sistema de arquivos.


In [ ]:
# ── Criar o GerenciadorCorpus ────────────────────────────────────────────────
gc = GerenciadorCorpus(str(DATA_DIR))

print("Gerenciador de corpus inicializado.")
print(f"  CSV: {gc.csv_path}")
print(f"  Diretório do corpus: {gc.corpus_dir}")
print(f"  Subpastas por país criadas: {[p.name for p in gc.corpus_dir.iterdir() if p.is_dir()]}")


### 2.1 Adicionando imagens históricas ao corpus

As 10 imagens a seguir cobrem todos os 5 países, os 3 regimes iconocráticos e múltiplos suportes materiais.  
São obras historicamente documentadas, em domínio público, amplamente estudadas na iconografia política.

| # | Obra | País | Ano | Suporte | Regime |
|---|------|------|-----|---------|--------|
| 1 | *La Liberté guidant le peuple* (Delacroix) | FR | 1830 | pintura | fundacional |
| 2 | *Marianne* — busto oficial da República Francesa | FR | 1848 | emblema | normativo |
| 3 | *Britannia* — penny de cobre vitoriana | UK | 1860 | moeda | normativo |
| 4 | *Britannia* — cartaz de recrutamento (WWI) | UK | 1914 | gravura | militar |
| 5 | *Columbia* — cartaz de recrutamento (Stahr) | USA | 1917 | gravura | militar |
| 6 | *Columbia* — frontispício do *Harper's Weekly* | USA | 1861 | gravura | fundacional |
| 7 | *Germania* — monumento de Rüdesheim am Rhein | DE | 1883 | monumento | normativo |
| 8 | *Justitia* — Fonte da Justiça de Frankfurt | DE | 1611 | monumento | normativo |
| 9 | *Efígie da República* (Henrique Bernardelli) | BR | 1889 | pintura | fundacional |
| 10 | *Selo Proclamação da República* | BR | 1889 | selo | fundacional |


In [ ]:
# ── Definição das 10 imagens históricas ─────────────────────────────────────

imagens_historicas = [

    # 1 — Delacroix, La Liberté guidant le peuple (1830)
    ImagemCorpus(
        Country="FR", Figure="Marianne", Support="pintura", Year="1830",
        Source_URL="https://commons.wikimedia.org/wiki/File:Eug%C3%A8ne_delacroix_-_la_libert%C3%A9_guidant_le_peuple.jpg",
        Rights="domínio_público", Regime_preliminary="fundacional",
        Description="La Liberté guidant le peuple — Delacroix. Alegoria da Liberdade conduzindo o povo nas Três Gloriosas (1830). Figura feminina com gorro frígio, seio exposto, portando bandeira tricolor e espingarda Brown Bess. Paradigma fundacional da iconografia republicana francesa.",
        Artist="Eugène Delacroix",
        Collection="Musée du Louvre, Paris (RF 129)",
    ),

    # 2 — Marianne, busto oficial pós-1848
    ImagemCorpus(
        Country="FR", Figure="Marianne", Support="emblema", Year="1848",
        Source_URL="https://commons.wikimedia.org/wiki/File:Marianne_buste.jpg",
        Rights="domínio_público", Regime_preliminary="normativo",
        Description="Marianne — busto oficial da II República Francesa. Figura feminina clássica com gorro frígio, drapeado, postura serena. Modelo normativo que passou a figurar em todas as prefeituras e repartições públicas francesas após 1870.",
        Artist="Desconhecido",
        Collection="Archives nationales, Paris",
    ),

    # 3 — Britannia, penny de cobre vitoriana (1860)
    ImagemCorpus(
        Country="UK", Figure="Britannia", Support="moeda", Year="1860",
        Source_URL="https://commons.wikimedia.org/wiki/File:British_Penny_1860.jpg",
        Rights="domínio_público", Regime_preliminary="normativo",
        Description="Britannia no reverso da penny de bronze vitoriana (1860). Figura sedente com elmo coríntio, tridente, escudo com Union Jack e ramo de oliveira. Iconografia imperial normativa estabilizada sob Vitória.",
        Artist="Leonard Charles Wyon",
        Collection="Royal Mint Museum, Llantrisant",
    ),

    # 4 — Britannia, cartaz de recrutamento WWI (1914)
    ImagemCorpus(
        Country="UK", Figure="Britannia", Support="gravura", Year="1914",
        Source_URL="https://commons.wikimedia.org/wiki/File:Britons_Lord_Kitchener_Wants_You.jpg",
        Rights="domínio_público", Regime_preliminary="militar",
        Description="Britannia em cartaz de recrutamento da I Guerra Mundial (1914). Variante militante: postura erguida, gestual imperativo, apelo à defesa do Império. Adaptação da iconografia normativa ao contexto de mobilização bélica.",
        Artist="Alfred Leete",
        Collection="Imperial War Museum, Londres (PST 2734)",
    ),

    # 5 — Columbia, cartaz de recrutamento (Stahr, 1917)
    ImagemCorpus(
        Country="USA", Figure="Columbia", Support="gravura", Year="1917",
        Source_URL="https://commons.wikimedia.org/wiki/File:Columbia_calls.jpg",
        Rights="domínio_público", Regime_preliminary="militar",
        Description="Columbia — cartaz 'Columbia Calls' de Paul Stahr (1917). Alegoria dos EUA em postura militante, vestida com a bandeira americana, gesticulando o chamado ao alistamento. Produzido para o Committee on Public Information (CPI).",
        Artist="Paul Stahr",
        Collection="Library of Congress, Washington D.C. (LC-USZC4-9897)",
    ),

    # 6 — Columbia, Harper's Weekly (1861)
    ImagemCorpus(
        Country="USA", Figure="Columbia", Support="gravura", Year="1861",
        Source_URL="https://commons.wikimedia.org/wiki/File:Columbia_Mourning_at_Lincoln.jpg",
        Rights="domínio_público", Regime_preliminary="fundacional",
        Description="Columbia no frontispício do Harper's Weekly (1861). Figura feminina clássica enlutada sobre o túmulo de Lincoln — alegoria sacrificial do regime fundacional americano. Reforça o nexo entre feminilidade, luto e fundação nacional.",
        Artist="John Chester Buttre",
        Collection="Library of Congress, Washington D.C.",
    ),

    # 7 — Germania, monumento de Rüdesheim (1883)
    ImagemCorpus(
        Country="DE", Figure="Germania", Support="monumento", Year="1883",
        Source_URL="https://commons.wikimedia.org/wiki/File:Niederwald_Denkmal_Germania.jpg",
        Rights="domínio_público", Regime_preliminary="normativo",
        Description="Niederwalddenkmal — monumento da Germania em Rüdesheim am Rhein (1883). Figura colossal (10,5 m) com coroa imperial, espada, escudo com águia. Erguida para celebrar a unificação alemã de 1871. Paradigma normativo do Second Reich.",
        Artist="Johannes Schilling",
        Collection="Niederwald, Rüdesheim am Rhein",
    ),

    # 8 — Justitia, Fonte da Justiça de Frankfurt (1611)
    ImagemCorpus(
        Country="DE", Figure="Justiça", Support="monumento", Year="1611",
        Source_URL="https://commons.wikimedia.org/wiki/File:Gerechtigkeitsbrunnen_Frankfurt.jpg",
        Rights="domínio_público", Regime_preliminary="normativo",
        Description="Gerechtigkeitsbrunnen (Fonte da Justiça), Frankfurt am Main (1611). Figura da Justitia com balança, espada e venda — tríade clássica dos atributos jurídicos. Centro histórico de Frankfurt; referência canônica na cultura jurídica alemã.",
        Artist="Johann Hocheisen",
        Collection="Römerberg, Frankfurt am Main",
    ),

    # 9 — Efígie da República, Henrique Bernardelli (1889)
    ImagemCorpus(
        Country="BR", Figure="República", Support="pintura", Year="1889",
        Source_URL="https://commons.wikimedia.org/wiki/File:Republica_brasileira_Bernardelli.jpg",
        Rights="domínio_público", Regime_preliminary="fundacional",
        Description="Efígie da República — Henrique Bernardelli (1889). Alegoria da República Brasileira: figura feminina com gorro frígio, drapeado azul-branco, postura serena e majestosa. Primeira iconografia oficial republicana do Brasil, inspirada na Marianne francesa e adaptada ao contexto tropical.",
        Artist="Henrique Bernardelli",
        Collection="Museu Nacional de Belas Artes, Rio de Janeiro",
    ),

    # 10 — Selo da Proclamação da República (1889)
    ImagemCorpus(
        Country="BR", Figure="República", Support="selo", Year="1889",
        Source_URL="https://commons.wikimedia.org/wiki/File:Stamp_Brazil_1889_100r.jpg",
        Rights="domínio_público", Regime_preliminary="fundacional",
        Description="Primeiro selo da República Brasileira (100 réis, 1889). Efígie feminina de perfil com gorro frígio — transposição direta da Marianne francesa para o suporte filatélico. Circulou imediatamente após a Proclamação de 15 de novembro de 1889.",
        Artist="Casa da Moeda do Brasil",
        Collection="Museu Postal Brasileiro, Brasília",
    ),
]

print(f"Total de imagens definidas: {len(imagens_historicas)}")
print("\nLista de imagens:")
for i, img in enumerate(imagens_historicas, 1):
    print(f"  [{i:2d}] {img.Country} | {img.Figure:<12} | {img.Year} | {img.Support:<25} | {img.Regime_preliminary}")


In [ ]:
# ── Adicionar as imagens ao corpus ───────────────────────────────────────────
print("Adicionando imagens ao corpus...")
print("─" * 65)

ids_gerados = []
for img in imagens_historicas:
    img_id = gc.adicionar_imagem(img)
    ids_gerados.append(img_id)
    print(f"  ✓ ID {img_id} | {img.Country} | {img.Figure:<12} | {img.Year}")

print("─" * 65)
print(f"\nTotal no corpus: {gc.contar_imagens()} imagens")
print(f"IDs gerados: {ids_gerados}")


### 2.2 Listagem e filtragem do corpus

O método `listar_imagens(filtros={...})` aceita qualquer combinação de campos como filtro.  
Abaixo demonstramos alguns usos típicos.


In [ ]:
# ── Listar todas as imagens ──────────────────────────────────────────────────
todas = gc.listar_imagens()
print(f"Total de registros no CSV: {len(todas)}\n")

# Exibir cabeçalho da tabela
print(f"{'ID':<6} {'País':<5} {'Figura':<14} {'Ano':<6} {'Suporte':<25} {'Regime':<13}")
print("─" * 75)
for img in todas:
    print(f"{img['Image_ID']:<6} {img['Country']:<5} {img['Figure']:<14} {img['Year']:<6} {img['Support']:<25} {img['Regime_preliminary']:<13}")


In [ ]:
# ── Filtrar por país ─────────────────────────────────────────────────────────
print("=== Imagens da FRANCE (FR) ===")
imgs_fr = gc.listar_imagens(filtros={"Country": "FR"})
for img in imgs_fr:
    print(f"  [{img['Image_ID']}] {img['Figure']} ({img['Year']}) — {img['Description'][:60]}...")

print()
print("=== Imagens do BRASIL (BR) ===")
imgs_br = gc.listar_imagens(filtros={"Country": "BR"})
for img in imgs_br:
    print(f"  [{img['Image_ID']}] {img['Figure']} ({img['Year']}) — {img['Description'][:60]}...")

print()
print("=== Imagens de regime FUNDACIONAL (todos os países) ===")
imgs_fund = gc.listar_imagens(filtros={"Regime_preliminary": "fundacional"})
for img in imgs_fund:
    print(f"  [{img['Image_ID']}] {img['Country']} | {img['Figure']} ({img['Year']})")


In [ ]:
# ── Buscar imagem por ID ─────────────────────────────────────────────────────
img = gc.buscar_por_id("0001")
if img:
    print("Detalhes da imagem 0001:")
    print("─" * 50)
    for campo, valor in img.items():
        if valor:
            # Truncar descrições longas para legibilidade
            valor_exib = valor[:90] + "..." if len(str(valor)) > 90 else valor
            print(f"  {campo:<22}: {valor_exib}")

# ── Demonstrar convenção de nomeação de arquivo ──────────────────────────────
print("\nConvenção de nomeação de arquivo:")
print("─" * 50)
for img_data in gc.listar_imagens():
    # Recriar ImagemCorpus para usar o método nome_arquivo()
    obj = ImagemCorpus(
        Image_ID=img_data["Image_ID"],
        Country=img_data["Country"],
        Figure=img_data["Figure"],
        Year=img_data["Year"],
        Support=img_data["Support"],
    )
    print(f"  → {obj.nome_arquivo('jpg')}")


### 2.3 Relatório de distribuição e checkpoints metodológicos

O método `imprimir_relatorio()` fornece:
- Distribuição por país, regime, suporte e figura
- Tabela cruzada país × regime
- **Validação de checkpoints**: verificação automática de metas metodológicas (mínimo de 300 imagens, pelo menos 50 por país, todos os regimes representados)

Os checkpoints são definidos a partir dos critérios de representatividade do corpus descritos no Capítulo 2 da tese. A banca pode verificar, em tempo de execução, se o corpus em construção satisfaz os requisitos metodológicos.


In [ ]:
# ── Relatório de distribuição completo ──────────────────────────────────────
gc.imprimir_relatorio()


> **Observação para a banca:** os checkpoints exibem ❌ porque estamos operando com apenas 10 imagens de demonstração.  
> Em produção, o corpus completo (300–500 imagens) preencherá todas as metas.  
> A célula de validação é executada a cada adição de imagem e pode ser integrada ao CI/CD do repositório.


In [ ]:
# ── Exportar bibliografia de fontes ─────────────────────────────────────────
gc.exportar_bibliografia_fontes("relatorio_fontes.txt")

# Mostrar conteúdo do arquivo gerado
fontes_path = DATA_DIR / "relatorio_fontes.txt"
with open(fontes_path, "r", encoding="utf-8") as f:
    print(f.read())


---
## 3. Os 10 Indicadores do Protocolo de Codificação

O sistema de codificação quantitativa é o coração metodológico do Atlas Iconocrático.  
Cada imagem do corpus é analisada segundo **10 indicadores operacionais**, agrupados em três dimensões:

### Dimensão I — Corporalidade e Género
| Indicador | Nome PT | Tipo | Escala |
|-----------|---------|------|--------|
| **FEI** | Índice de Corporalidade Feminina | Ordinal | 0–3 |
| **CII** | Índice de Ideal Clássico | Ordinal | 0–3 |
| **PRI** | Índice de Postura e Domínio | Categórico | serena / militante / sacrificial |

### Dimensão II — Simbolismo e Conflito
| Indicador | Nome PT | Tipo | Escala |
|-----------|---------|------|--------|
| **SMI** | Índice de Material Simbólico | Ordinal | 0–3 |
| **SMS** | Pontuação de Significado Simbólico | Categórico | serena / militante / sacrificial |
| **AMCP** | Perfil de Armamento e Crise | Ordinal | 0–3 |
| **MVI** | Índice de Maternidade-Vulnerabilidade | Ordinal | 0–3 |

### Dimensão III — Raça e Identidade
| Indicador | Nome PT | Tipo | Escala |
|-----------|---------|------|--------|
| **WI** | Índice de Branquitude | Ordinal | 0–3 |
| **RI** | Idealização Racial | Booleano | 0 / 1 |
| **AI** | Índice Alegoria-Biografia | Ordinal | 0–3 |

---

### Painéis analíticos da tese

Os 10 indicadores alimentam **7 painéis analíticos** que estruturam os capítulos da tese:

| Painel | Título | Indicadores principais |
|--------|--------|------------------------|
| I | Mães Fundacionais | FEI, PRI, MVI |
| II | Justiça como Templo | CII, SMI |
| III | Corpos em Armas | PRI, SMS, AMCP |
| IV | República Branca | CII, WI, RI |
| V | Ubiquidade Íntima | FEI |
| VI | Damas de Ferro | AI |
| VII | Justiça Contestada | SMS |


In [ ]:
# ── Descrição de cada indicador individualmente ──────────────────────────────
# Começamos com o FEI, indicador central da tese

descrever_indicador("FEI")


In [ ]:
descrever_indicador("AMCP")


In [ ]:
descrever_indicador("WI")


In [ ]:
# ── Descrição de TODOS os 10 indicadores ────────────────────────────────────
descrever_todos_indicadores()


### 3.1 Árvores de decisão

Para cada indicador com potencial de ambiguidade no processo de codificação, o módulo fornece **árvores de decisão** que guiam o codificador passo a passo. Elas integram o manual de codificação e são o principal instrumento de treinamento dos codificadores secundários (necessários para o cálculo do kappa).


In [ ]:
# ── Árvore de decisão: FEI ───────────────────────────────────────────────────
print(arvore_decisao("FEI"))


In [ ]:
# ── Árvore de decisão: AMCP ─────────────────────────────────────────────────
print(arvore_decisao("AMCP"))


In [ ]:
# ── Árvore de decisão: WI ───────────────────────────────────────────────────
print(arvore_decisao("WI"))


In [ ]:
# ── Árvore de decisão: MVI ──────────────────────────────────────────────────
print(arvore_decisao("MVI"))


In [ ]:
# ── Exportar manual de codificação completo como JSON ───────────────────────
manual_path = str(DATA_DIR / "manual_codificacao.json")
gerar_manual_codificacao(manual_path)

# Visualizar a estrutura do manual
import json
with open(manual_path, "r", encoding="utf-8") as f:
    manual = json.load(f)

print("\nEstrutura do manual_codificacao.json:")
print(f"  Número de indicadores: {len(manual)}")
print()
for codigo, info in manual.items():
    print(f"  ▸ {codigo}: {info['nome_pt']} [{info['tipo']}] escala={info['escala']}")


---
## 4. Codificação das Imagens

### Protocolo de codificação

Cada imagem do corpus é codificada individualmente pelo pesquisador principal (COD1) e, para a amostra de confiabilidade, por um segundo codificador treinado (COD2). A codificação segue este protocolo:

1. **Observação da imagem** — sem consultar codificações anteriores
2. **Aplicação sequencial dos 10 indicadores** — na ordem FEI → CII → PRI → SMI → SMS → AMCP → MVI → WI → RI → AI
3. **Consulta às árvores de decisão** em casos de ambiguidade
4. **Registro de observações qualitativas** no campo `Notes`
5. **Validação automática** pelo sistema antes do salvamento

### Justificativa das codificações

A tabela abaixo sintetiza as justificativas iconográficas para cada imagem do corpus de demonstração:

| ID | Obra | FEI | CII | PRI | AMCP | MVI | WI | RI | AI | Justificativa |
|----|------|-----|-----|-----|------|-----|----|----|----|---------------|
| 0001 | Delacroix (1830) | 3 | 1 | militante | 2 | 2 | 1 | 0 | 1 | Seio exposto, gorro frígio (não greco-romano), lança + espingarda, postura de ataque |
| 0002 | Marianne busto | 1 | 2 | serena | 0 | 0 | 2 | 1 | 1 | Corpo coberto, peplos + gorro frígio, sem armas, branquitude neoclássica |
| 0003 | Britannia moeda | 1 | 3 | serena | 1 | 0 | 2 | 1 | 1 | Drapeado clássico completo, elmo coríntio + escudo (defensivo), perfil greco-romano |
| 0004 | Britannia WWI | 1 | 2 | militante | 1 | 0 | 2 | 1 | 1 | Gesto imperativo, escudo/elmo mas sem arma ofensiva, transição normativo→militar |
| 0005 | Columbia Stahr | 1 | 1 | militante | 0 | 0 | 2 | 1 | 1 | Bandeira americana, gestual de chamada, sem armas, branquitude americana |
| 0006 | Columbia Harper's | 1 | 1 | sacrificial | 0 | 3 | 2 | 1 | 1 | Luto sobre túmulo, sacrifício fundacional, sem armas, figura enlutada |
| 0007 | Germania (1883) | 1 | 2 | serena | 2 | 0 | 2 | 1 | 1 | Figura colossal erguida, espada + escudo, coroa imperial, branquitude racial explícita |
| 0008 | Justitia Frankfurt | 1 | 3 | serena | 2 | 0 | 2 | 1 | 0 | Tríade clássica (balança+espada+venda), iconografia greco-romana completa, alegoria pura |
| 0009 | República BR | 1 | 1 | serena | 0 | 0 | 1 | 0 | 1 | Gorro frígio (não greco-romano puro), drapeado, sem armas, traços mesclados |
| 0010 | Selo BR (1889) | 0 | 1 | serena | 0 | 0 | 1 | 0 | 1 | Perfil filatélico estilizado, mínima corporalidade, gorro frígio isolado |


In [ ]:
# ── Criar o GerenciadorCodificacao ──────────────────────────────────────────
gcod = GerenciadorCodificacao(str(DATA_DIR))

print("Gerenciador de codificação inicializado.")
print(f"  Diretório base: {gcod.base}")


In [ ]:
# ── Codificações das 10 imagens históricas ───────────────────────────────────

codificacoes = [

    # 0001 — La Liberté guidant le peuple (Delacroix, 1830)
    # FEI=3: seio esquerdo exposto, nudez parcial explícita
    # CII=1: gorro frígio (elemento pós-clássico), não greco-romano puro
    # PRI=militante: marcha ativa sobre barricada, gestual de ataque
    # SMI=2: bandeira tricolor + espingarda (2 atributos)
    # SMS=militante: simbologia de revolução e guerra civil
    # AMCP=2: espingarda (arma ofensiva); baioneta visível
    # MVI=2: seio exposto — amamentação simbólica da nação revolucionária
    # WI=1: traços europeus mas representação popular/híbrida, não neoclássica pura
    # RI=0: não corresponde ao ideal clássico branco greco-romano
    # AI=1: alegoria nomeada genérica ("Liberté/Marianne")
    CodificacaoImagem(
        Image_ID="0001", Coder_ID="COD1",
        FEI=3, CII=1, PRI="militante",
        SMI=2, SMS="militante",
        AMCP=2, MVI=2, WI=1, RI=0, AI=1,
        Notes="Seio esquerdo exposto; gorro frígio (não elmo coríntio); espingarda Brown Bess + baioneta; marcha sobre corpos na barricada; traços populares, não neoclássicos.",
    ),

    # 0002 — Marianne, busto oficial (1848)
    # FEI=1: corpo sugerido por drapeado; busto — sem nudez
    # CII=2: peplos/drapeado clássico + gorro frígio — predominância clássica
    # PRI=serena: busto estático, postura contemplativa
    # SMI=1: apenas gorro frígio como atributo simbólico
    # SMS=serena: lei/ordem — símbolo normativo da República
    # AMCP=0: sem armas
    # MVI=0: sem marcas maternais
    # WI=2: branquitude neoclássica explícita, pele clara idealizada
    # RI=1: ideal clássico branco convergente
    # AI=1: alegoria nomeada genérica ("Marianne")
    CodificacaoImagem(
        Image_ID="0002", Coder_ID="COD1",
        FEI=1, CII=2, PRI="serena",
        SMI=1, SMS="serena",
        AMCP=0, MVI=0, WI=2, RI=1, AI=1,
        Notes="Busto normativo; drapeado clássico + gorro frígio; sem nudez; postura estática contemplativa; padrão de todas as prefeituras francesas após 1870.",
    ),

    # 0003 — Britannia, penny vitoriana (1860)
    # FEI=1: corpo completamente coberto por armadura/drapeado
    # CII=3: iconografia greco-romana canônica completa (elmo coríntio, peplos, tridente, escudo)
    # PRI=serena: figura sedente, postura de governo pacífico imperial
    # SMI=3: balança + tridente + escudo + ramo de oliveira (4+ atributos)
    # SMS=serena: lei/ordem — estabilidade imperial britânica
    # AMCP=1: escudo com Union Jack (arma defensiva)
    # MVI=0: sem marcas maternais
    # WI=2: branquitude neoclássica explícita em perfil de moeda
    # RI=1: ideal clássico branco convergente
    # AI=1: alegoria nomeada genérica ("Britannia")
    CodificacaoImagem(
        Image_ID="0003", Coder_ID="COD1",
        FEI=1, CII=3, PRI="serena",
        SMI=3, SMS="serena",
        AMCP=1, MVI=0, WI=2, RI=1, AI=1,
        Notes="Clássico canônico: elmo coríntio + tridente + escudo Union Jack + ramo oliveira; figura sedente; perfil neoclássico em relevo de moeda vitoriana.",
    ),

    # 0004 — Britannia, cartaz de recrutamento WWI (1914)
    # FEI=1: corpo coberto por drapeado + elementos de armadura
    # CII=2: elmo + drapeado clássico mas adaptação moderna do cartaz
    # PRI=militante: postura erguida, gesto imperativo de chamada ao alistamento
    # SMI=2: escudo + gestual imperativo (2 atributos simbólicos)
    # SMS=militante: mobilização bélica, defesa do Império
    # AMCP=1: escudo (arma defensiva); sem arma ofensiva
    # MVI=0: sem marcas maternais
    # WI=2: branquitude explícita em poster de propaganda
    # RI=1: ideal clássico branco convergente
    # AI=1: alegoria nomeada genérica ("Britannia")
    CodificacaoImagem(
        Image_ID="0004", Coder_ID="COD1",
        FEI=1, CII=2, PRI="militante",
        SMI=2, SMS="militante",
        AMCP=1, MVI=0, WI=2, RI=1, AI=1,
        Notes="Gesto imperativo de chamada ao alistamento; escudo mas sem arma ofensiva; transição normativo→militar; adaptação do modelo moeda para cartaz de propaganda WWI.",
    ),

    # 0005 — Columbia, cartaz Stahr (1917)
    # FEI=1: corpo coberto pela bandeira americana
    # CII=1: elementos mistos — coroa radiada (americana, não clássica) + drapeado
    # PRI=militante: gestual ativo de chamada ao alistamento
    # SMI=2: bandeira + gestual (2 atributos)
    # SMS=militante: mobilização para a I Guerra Mundial
    # AMCP=0: sem armas (apelo moral, não bélico direto)
    # MVI=0: sem marcas maternais
    # WI=2: branquitude explícita americana em cartaz de propaganda
    # RI=1: ideal clássico branco convergente
    # AI=1: alegoria nomeada genérica ("Columbia")
    CodificacaoImagem(
        Image_ID="0005", Coder_ID="COD1",
        FEI=1, CII=1, PRI="militante",
        SMI=2, SMS="militante",
        AMCP=0, MVI=0, WI=2, RI=1, AI=1,
        Notes="Bandeira americana como vestimenta; gesto de chamada ao alistamento; coroa radiada americana (não greco-romana); sem armas; CPI/Committee on Public Information.",
    ),

    # 0006 — Columbia, Harper's Weekly (1861) — luto sobre Lincoln
    # FEI=1: corpo coberto, postura enlutada
    # CII=1: drapeado com elementos americanos, não greco-romano puro
    # PRI=sacrificial: postura de luto, figura curvada sobre túmulo
    # SMI=2: bandeira + coroa funerária (2 atributos)
    # SMS=sacrificial: fundação pelo trauma — luto fundacional americano
    # AMCP=0: sem armas — figura exclusivamente sacrificial
    # MVI=3: corpo enlutado/sacrificial — corpo ferido simbolicamente pela morte do Pai da Nação
    # WI=2: branquitude neoclássica em gravura vitoriana americana
    # RI=1: ideal clássico branco convergente
    # AI=1: alegoria nomeada genérica ("Columbia")
    CodificacaoImagem(
        Image_ID="0006", Coder_ID="COD1",
        FEI=1, CII=1, PRI="sacrificial",
        SMI=2, SMS="sacrificial",
        AMCP=0, MVI=3, WI=2, RI=1, AI=1,
        Notes="Postura de luto sobre túmulo de Lincoln; sacrifício fundacional americano; sem armas; modo sacrificial que converte o assassinato em fundação mítica da nação.",
    ),

    # 0007 — Germania, monumento de Rüdesheim (1883)
    # FEI=1: corpo completamente coberto por armadura romana adaptada
    # CII=2: coroa imperial + armadura + escudo — predominância clássica adaptada
    # PRI=serena: figura erguida mas estática, postura monumental de governo
    # SMI=3: coroa imperial + espada + escudo + placa com inscrições (4 atributos)
    # SMS=serena: ordem/lei — celebração da unificação alemã, normatividade imperial
    # AMCP=2: espada (arma ofensiva) + escudo
    # MVI=0: sem marcas maternais
    # WI=2: branquitude explícita — escultura neoclássica em bronze/pedra
    # RI=1: ideal clássico branco convergente com ideologia pangermanista
    # AI=1: alegoria nomeada genérica ("Germania")
    CodificacaoImagem(
        Image_ID="0007", Coder_ID="COD1",
        FEI=1, CII=2, PRI="serena",
        SMI=3, SMS="serena",
        AMCP=2, MVI=0, WI=2, RI=1, AI=1,
        Notes="Figura colossal (10,5m); coroa imperial + espada erguida + escudo com águia; erguida após unificação de 1871; ideologia do Second Reich; branquitude racial explícita.",
    ),

    # 0008 — Justitia, Fonte da Justiça de Frankfurt (1611)
    # FEI=1: figura coberta, drapeado; escultura de bronze
    # CII=3: iconografia greco-romana canônica completa (tríade balança+espada+venda)
    # PRI=serena: figura estática sobre fonte, postura de autoridade legal
    # SMI=3: balança + espada + venda (tríade clássica = 3 atributos)
    # SMS=serena: lei/ordem — símbolo jurídico por excelência
    # AMCP=2: espada (arma ofensiva) como símbolo da justiça punitiva
    # MVI=0: sem marcas maternais
    # WI=2: branquitude neoclássica em escultura europeia do século XVII
    # RI=1: ideal clássico branco convergente
    # AI=0: alegoria pura — sem nome individual, símbolo universal da Justiça
    CodificacaoImagem(
        Image_ID="0008", Coder_ID="COD1",
        FEI=1, CII=3, PRI="serena",
        SMI=3, SMS="serena",
        AMCP=2, MVI=0, WI=2, RI=1, AI=0,
        Notes="Tríade clássica completa: balança + espada + venda; iconografia greco-romana canônica; alegoria pura (AI=0) — sem nome individual, sem inspiração em pessoa real.",
    ),

    # 0009 — Efígie da República, Bernardelli (1889)
    # FEI=1: corpo coberto por drapeado
    # CII=1: gorro frígio + drapeado, mas sem elementos greco-romanos puros — mistura brasileira
    # PRI=serena: postura majestosa estática
    # SMI=1: apenas gorro frígio como atributo simbólico
    # SMS=serena: lei/ordem — fundação republicana pacífica
    # AMCP=0: sem armas
    # MVI=0: sem marcas maternais
    # WI=1: traços clássicos sugeridos mas representação com ambiguidade racial — Brasil mestiço
    # RI=0: não corresponde ao ideal clássico branco puro — traços misturados
    # AI=1: alegoria nomeada genérica ("República")
    CodificacaoImagem(
        Image_ID="0009", Coder_ID="COD1",
        FEI=1, CII=1, PRI="serena",
        SMI=1, SMS="serena",
        AMCP=0, MVI=0, WI=1, RI=0, AI=1,
        Notes="Gorro frígio + drapeado; postura majestosa; RI=0 porque traços não seguem o ideal greco-romano puro — representação com ambiguidade racial própria do contexto brasileiro (1889).",
    ),

    # 0010 — Selo da Proclamação da República (1889)
    # FEI=0: perfil filatélico estilizado — corporalidade mínima, abstrata
    # CII=1: gorro frígio visível como único elemento clássico/simbólico
    # PRI=serena: perfil estático — convenção filatélica
    # SMI=1: apenas gorro frígio
    # SMS=serena: lei/ordem — emblema de Estado
    # AMCP=0: sem armas
    # MVI=0: sem marcas maternais
    # WI=1: traços sugeridos em perfil de selo — representação estilizada
    # RI=0: perfil simplificado não projeta ideal racial explícito
    # AI=1: alegoria nomeada genérica ("República")
    CodificacaoImagem(
        Image_ID="0010", Coder_ID="COD1",
        FEI=0, CII=1, PRI="serena",
        SMI=1, SMS="serena",
        AMCP=0, MVI=0, WI=1, RI=0, AI=1,
        Notes="FEI=0: perfil filatélico altamente estilizado, sem corporalidade reconhecível. Gorro frígio como único atributo simbólico. Convenção do suporte (selo) impõe economia icônica.",
    ),
]

print(f"Total de codificações preparadas: {len(codificacoes)}")


In [ ]:
# ── Salvar codificações em lote ──────────────────────────────────────────────
gcod.salvar_lote(codificacoes)

print(f"\nArquivo gerado: {gcod._csv_path('COD1')}")


In [ ]:
# ── Relatório de progresso ───────────────────────────────────────────────────
gcod.imprimir_progresso("COD1", total_corpus=500)

print()
print("Nota: em produção, o corpus completo terá 300–500 imagens.")
print("A barra de progresso reflete o andamento real da codificação.")


In [ ]:
# ── Consolidar dataset final ─────────────────────────────────────────────────
csv_consolidado = gcod.consolidar_dataset(coder_id_principal="COD1")
print(f"\nDataset consolidado salvo em: {csv_consolidado}")


---
## 5. Validação e Perfil das Codificações

### 5.1 Perfis iconográficos

O método `perfil_resumido()` fornece uma linha compacta com os 10 valores, útil para revisão rápida e discussão com a banca.


In [ ]:
# ── Perfis resumidos de todas as codificações ────────────────────────────────
print("Perfis iconográficos — Corpus de demonstração")
print("=" * 75)
print(f"{'ID':<6} {'Obra':<30} {'Perfil'}")
print("─" * 75)

obras = [
    "Delacroix — La Liberté (1830)",
    "Marianne, busto oficial (1848)",
    "Britannia, penny vitoriana (1860)",
    "Britannia, cartaz WWI (1914)",
    "Columbia, cartaz Stahr (1917)",
    "Columbia, Harper's Weekly (1861)",
    "Germania, monumento (1883)",
    "Justitia, Frankfurt (1611)",
    "República BR, Bernardelli (1889)",
    "Selo Proclamação República (1889)",
]

for cod, obra in zip(codificacoes, obras):
    print(f"{cod.Image_ID:<6} {obra:<30} {cod.perfil_resumido()}")


### 5.2 Validação de codificações

O sistema rejeita codificações inválidas **antes de salvá-las**. A validação verifica:
- Que todos os 10 indicadores foram preenchidos (sem valor `-1` / string vazia)
- Que os valores ordinais estão em `{0, 1, 2, 3}`
- Que os valores booleanos estão em `{0, 1}`
- Que os valores categóricos estão em `{"serena", "militante", "sacrificial"}`

A célula abaixo demonstra o comportamento com dados inválidos.


In [ ]:
# ── Demonstração de validação: dados inválidos ───────────────────────────────
print("=== Tentativa de codificação com dados INVÁLIDOS ===\n")

cod_invalida = CodificacaoImagem(
    Image_ID="9999",
    Coder_ID="",           # ← Coder_ID faltando
    FEI=5,                 # ← FEI fora do range (deve ser 0–3)
    CII=2,
    PRI="agressiva",       # ← PRI inválido (deve ser serena/militante/sacrificial)
    SMI=1,
    SMS="serena",
    AMCP=2,
    MVI=0,
    WI=-1,                 # ← WI negativo (inválido)
    RI=2,                  # ← RI deve ser 0 ou 1
    AI=1,
)

erros = cod_invalida.validar()
print(f"Erros encontrados ({len(erros)}):")
for e in erros:
    print(f"  ✗ {e}")

print("\n→ A codificação acima NÃO seria salva.")
print("→ O sistema bloqueia o salvamento e retorna a lista de erros.")


In [ ]:
# ── Demonstração de validação: dados VÁLIDOS ─────────────────────────────────
print("=== Codificação com dados VÁLIDOS ===\n")

cod_valida = CodificacaoImagem(
    Image_ID="0001", Coder_ID="COD2",
    FEI=3, CII=1, PRI="militante",
    SMI=2, SMS="militante",
    AMCP=2, MVI=2, WI=1, RI=0, AI=1,
    Notes="Revisão do codificador 2 — confirma codificação primária.",
)

erros = cod_valida.validar()
if not erros:
    print("✅ Codificação válida — pronta para salvamento.")
    print(f"   Perfil: {cod_valida.perfil_resumido()}")
else:
    print(f"❌ Erros: {erros}")


### 5.3 Visualização do dataset consolidado

O arquivo `corpus_codificado.csv` contém todas as codificações do codificador principal.  
Ele é o input direto para o **Notebook 2** (análise estatística) e o **Notebook 3** (kappa + Iconclass).


In [ ]:
# ── Visualizar o dataset consolidado ────────────────────────────────────────
import csv

csv_path = DATA_DIR / "corpus_codificado.csv"
with open(csv_path, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print(f"corpus_codificado.csv — {len(rows)} registros")
print()

# Cabeçalho
campos_exibir = ["Image_ID", "Coder_ID", "FEI", "CII", "PRI", "SMI", "SMS", "AMCP", "MVI", "WI", "RI", "AI"]
header = " │ ".join(f"{c:<7}" for c in campos_exibir)
print(header)
print("─" * len(header))

for row in rows:
    linha = " │ ".join(f"{row.get(c, ''):<7}" for c in campos_exibir)
    print(linha)


In [ ]:
# ── Estatísticas descritivas básicas dos indicadores ordinais ────────────────
indicadores_ordinais = ["FEI", "CII", "SMI", "AMCP", "MVI", "WI", "AI"]

print("Estatísticas descritivas básicas (corpus de demonstração)")
print("=" * 55)
print(f"{'Indicador':<10} {'N':>4} {'Mín':>5} {'Máx':>5} {'Média':>8} {'Moda':>6}")
print("─" * 55)

for ind in indicadores_ordinais:
    valores = [int(row[ind]) for row in rows if row.get(ind) not in ("", None)]
    n = len(valores)
    if n == 0:
        continue
    minv = min(valores)
    maxv = max(valores)
    media = sum(valores) / n
    # Moda simples
    contagem = {v: valores.count(v) for v in set(valores)}
    moda = max(contagem, key=contagem.get)
    print(f"  {ind:<8} {n:>4} {minv:>5} {maxv:>5} {media:>8.2f} {moda:>6}")

print()
print("Distribuição de PRI (categórico):")
pri_vals = [row["PRI"] for row in rows if row.get("PRI")]
for cat in ("serena", "militante", "sacrificial"):
    n = pri_vals.count(cat)
    pct = n / len(pri_vals) * 100
    print(f"  {cat:<12}: {n} ({pct:.0f}%)")

print()
print("Distribuição de SMS (categórico):")
sms_vals = [row["SMS"] for row in rows if row.get("SMS")]
for cat in ("serena", "militante", "sacrificial"):
    n = sms_vals.count(cat)
    pct = n / len(sms_vals) * 100
    print(f"  {cat:<12}: {n} ({pct:.0f}%)")


---
## 6. Próximos Passos

Este tutorial cobriu a infraestrutura de dados do Atlas Iconocrático: gestão do corpus e codificação. Os notebooks seguintes aprofundam a análise:

---

### Notebook 2 — Análise Estatística (estatistica.py)

**Arquivo:** `02_analise_estatistica.ipynb`

Conteúdo:
- Estatística descritiva completa dos 10 indicadores (médias, desvios-padrão, distribuições)
- Testes de hipótese H1–H5:
  - **H1:** FEI varia significativamente entre países
  - **H2:** Regimes militares apresentam AMCP mais elevado
  - **H3:** WI correlaciona com CII (hipótese da "branquitude clássica")
  - **H4:** PRI militante concentra-se nos regimes fundacional e militar
  - **H5:** AI ≥ 2 é excepcional — alegorização de pessoas reais é rara
- Visualizações: boxplots, heatmaps de correlação, mapas de dispersão
- Uso do módulo `atlas_iconocratico.estatistica`

```python
from atlas_iconocratico.estatistica import AnalisadorEstatistico
ae = AnalisadorEstatistico(str(DATA_DIR))
ae.descritiva_completa()
ae.testar_h1_fei_por_pais()
```

---

### Notebook 3 — Confiabilidade Inter-Codificadores e Iconclass (kappa.py + iconclass_.py)

**Arquivo:** `03_kappa_iconclass.ipynb`

Conteúdo:
- Cálculo do **Kappa de Cohen** para cada um dos 10 indicadores
  - Kappa ponderado para indicadores ordinais (FEI, CII, SMI, AMCP, MVI, WI, AI)
  - Kappa simples para booleano (RI) e categóricos (PRI, SMS)
  - Meta metodológica: κ ≥ 0.70 para todos os indicadores
- **Análise de discordâncias**: identificação das imagens mais problemáticas
- **Integração Iconclass**: enriquecimento das codificações com o vocabulário controlado BARTSCH/RKD
  - Mapeamento FEI → notação Iconclass (ex.: `11G` = female personifications)
  - Exportação de `corpus_codificado_iconclass.csv`
- Uso dos módulos `atlas_iconocratico.kappa` e `atlas_iconocratico.iconclass_`

```python
from atlas_iconocratico.kappa import CalculadorKappa
ck = CalculadorKappa(str(DATA_DIR))
ck.calcular_todos_indicadores("COD1", "COD2")
ck.relatorio_kappa()
```

---

### Notebook 4 — Matriz de Evidências (matriz.py)

**Arquivo:** `04_matriz_evidencias.ipynb`

Conteúdo:
- Construção da **Matriz de Evidências** dos 7 painéis analíticos
- Exportação final para análise qualitativa
- Uso do módulo `atlas_iconocratico.matriz`

---

> **Referência metodológica:** Para detalhes completos sobre a operacionalização dos indicadores e os fundamentos teóricos do regime iconocrático, consultar o Capítulo 2 da tese: *"Contrato Sexual Visual: A Semiótica do Corpo Feminino nos Suportes Jurídicos Oficiais"*.
